# NumPy interview coding lab

Install a compatible NumPy build into the notebook kernel environment if missing. Exercises use synthetic CPU arrays; no other numerical framework or downloads are needed.

Use only the synthetic fixtures supplied here. No private data is included. This notebook never installs packages automatically. Replace each `...`, run the exercise cell, then its separate test cell. Failures before completion are intentional. Tests cover selected cases, not every production input. Reference solutions are text at the end and are not executed automatically.

Original exercises cover topics from the full [NumPy interview-preparation repository](https://github.com/rohanmistry231/Numpy-Interview-Preparation) (MIT-licensed). Its source examples are not reproduced.


In [ ]:
import numpy as np
print('NumPy:', np.__version__)


## 01 / INDEXING & OWNERSHIP: Distinguish slices, paired indices, and rectangular selection

Select from a fixed [4,5] array while preserving the difference between a basic-slice view and advanced-index copies.

### Hints
- **Rows 1 and 3, columns 1 and 3, with basic slices:** Stops are exclusive. Both dimensions use basic slicing, which shares storage here.
- **Select coordinate pairs (0,1) and (3,4):** Equal-shaped integer index arrays are paired elementwise, not expanded into a grid.
- **Select every combination of rows [0,3] and columns [1,4]:** ix_ creates broadcastable index arrays for an outer rectangular selection.

### Expected behavior
For arange(20).reshape(4,5): slice [[6,8],[16,18]], paired [1,19], rectangle [[1,4],[16,19]]. Only the basic slice shares array storage.


In [ ]:
import numpy as np

def select_regions(a: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if a.shape != (4, 5):
        raise ValueError("This fixture requires shape (4,5)")
    sliced = ...
    paired = ...
    rectangle = ...
    return sliced, paired, rectangle


In [ ]:
a = np.arange(20).reshape(4, 5)
sliced, paired, rectangle = select_regions(a)
np.testing.assert_array_equal(sliced, [[6, 8], [16, 18]])
np.testing.assert_array_equal(paired, [1, 19])
np.testing.assert_array_equal(rectangle, [[1, 4], [16, 19]])
assert np.shares_memory(a, sliced)
assert not np.shares_memory(a, paired)
assert not np.shares_memory(a, rectangle)
sliced[0, 0] = -9
assert a[1, 1] == -9
rectangle[0, 0] = -7
assert a[0, 1] == 1
print('PASS: values, shapes, pairing, outer selection, view/copy behavior')


## 02 / BROADCASTING & STATISTICS: Standardize features with the correct reduction axis

Fit per-feature training means/scales and preserve dimensions so [1,D] statistics broadcast over [N,D] samples.

### Hints
- **Mean across samples, keeping a singleton sample axis:** Rows are samples. Reduce axis 0 to obtain one statistic per feature.
- **Population standard deviation with the same shape:** This exercise uses ddof=0. Constant features intentionally use scale 1 in the given policy.
- **Center and scale without repeating statistics:** Broadcast [1,D] across [N,D]; do not tile the statistics into a full matrix.

### Expected behavior
For [[1,2,7],[3,4,7],[5,6,7]], mean is [[3,4,7]]. The constant third feature becomes zero; other standardized columns have mean 0 and population std 1.


In [ ]:
import numpy as np

def fit_standardize(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or 0 in x.shape or not np.isfinite(x).all():
        raise ValueError("Expected a finite nonempty [samples,features] array")
    mean = ...
    scale = ...
    scale = np.where(scale == 0, 1.0, scale)
    standardized = ...
    return standardized, mean, scale


In [ ]:
x = np.array([[1., 2., 7.], [3., 4., 7.], [5., 6., 7.]])
original = x.copy()
z, mean, scale = fit_standardize(x)
np.testing.assert_allclose(mean, [[3, 4, 7]])
assert mean.shape == scale.shape == (1, 3)
np.testing.assert_allclose(z.mean(axis=0), 0, atol=1e-12)
np.testing.assert_allclose(z[:, :2].std(axis=0), 1)
np.testing.assert_array_equal(z[:, 2], 0)
np.testing.assert_array_equal(x, original)
for invalid in [np.empty((0, 3)), np.array([[np.nan]])]:
    try:
        fit_standardize(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid data must fail')
print('PASS: axis, shape, scaling, constant feature, unchanged input, invalid data')


## 03 / UFUNCS & NUMERICS: Compute stable row probabilities in NumPy

Use reductions, broadcasting, and exp without overflowing on large finite logits.

### Hints
- **Subtract each row's maximum:** Normalize the last axis independently; preserve it for broadcasting.
- **Apply the exponential ufunc:** exp is an elementwise ufunc; np.vectorize is unnecessary.
- **Divide by each row's sum:** The last-axis sum must keep its dimension.

### Expected behavior
Rows [1000,1001,1002] and [-1000,-999,-998] yield the same probabilities, approximately [0.0900,0.2447,0.6652].


In [ ]:
import numpy as np

def row_softmax(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    if logits.ndim < 1 or 0 in logits.shape or not np.isfinite(logits).all():
        raise ValueError("Expected finite, nonempty logits")
    shifted = ...
    weights = ...
    return ...


In [ ]:
x = np.array([[1000., 1001., 1002.], [-1000., -999., -998.]])
p = row_softmax(x)
assert np.isfinite(p).all()
np.testing.assert_allclose(p.sum(axis=-1), 1)
np.testing.assert_allclose(p[0], p[1])
np.testing.assert_allclose(p[0], [0.0900305732, 0.2447284711, 0.6652409558])
np.testing.assert_allclose(row_softmax(np.zeros((2, 3, 4))), 0.25)
for invalid in [np.empty((0, 2)), np.array([np.nan])]:
    try:
        row_softmax(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid logits must fail')
print('PASS: large logits, shift invariance, normalization, batch axes, invalid inputs')


## 04 / VECTORIZATION & MEMORY: Vectorize pairwise squared distances

Map [N,D] points and [K,D] centers to [N,K] distances while tracking the [N,K,D] intermediate.

### Hints
- **Create point-center differences:** [N,1,D] minus [1,K,D] broadcasts to [N,K,D].
- **Sum squared differences over features:** Reduce D, not N or K.

### Expected behavior
Points [[0,0],[3,4]] and centers [[0,0],[0,4]] produce [[0,16],[25,9]].


In [ ]:
import numpy as np

def squared_distances(x: np.ndarray, centers: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    centers = np.asarray(centers, dtype=np.float64)
    if (x.ndim != 2 or centers.ndim != 2 or 0 in x.shape or 0 in centers.shape
            or x.shape[1] != centers.shape[1]
            or not np.isfinite(x).all() or not np.isfinite(centers).all()):
        raise ValueError("Expected finite nonempty [N,D] and [K,D] arrays")
    difference = ...
    return ...


In [ ]:
x = np.array([[0, 0], [3, 4]], dtype=np.int16)
c = np.array([[0, 0], [0, 4]], dtype=np.int16)
d = squared_distances(x, c)
np.testing.assert_array_equal(d, [[0, 16], [25, 9]])
np.testing.assert_array_equal(np.diag(squared_distances(x, x)), 0)
assert d.shape == (2, 2) and d.dtype == np.float64
try:
    squared_distances(x, np.ones((3, 5)))
except ValueError:
    pass
else:
    raise AssertionError('Feature dimensions must match')
print('PASS: distances, axes, dtype, self-distance, dimension mismatch')


## 05 / LINEAR ALGEBRA: Fit a line without explicitly inverting a matrix

Construct a design matrix with an intercept and use a least-squares solver, including a rank-deficient fixture.

### Hints
- **Create [intercept column, x column]:** The coefficient order is intercept, then slope.
- **Solve least squares and select the coefficient vector:** lstsq returns coefficients plus diagnostics; do not form inverse(X.T@X).

### Expected behavior
For x=[0,1,2] and y=[1,3,5], coefficients are [1,2]. With x=[1,1,1], y=[2,2,2], a minimum-norm solution still predicts 2.


In [ ]:
import numpy as np

def fit_line(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    if (x.ndim != 1 or y.shape != x.shape or x.size < 2
            or not np.isfinite(x).all() or not np.isfinite(y).all()):
        raise ValueError("Expected matching finite vectors with at least two samples")
    design = ...
    coefficients = ...
    return coefficients


In [ ]:
np.testing.assert_allclose(fit_line(np.array([0, 1, 2]), np.array([1, 3, 5])), [1, 2], atol=1e-12)
coefficients = fit_line(np.ones(3), np.full(3, 2.))
np.testing.assert_allclose(np.column_stack([np.ones(3), np.ones(3)]) @ coefficients, 2, atol=1e-12)
np.testing.assert_allclose(coefficients, [1, 1], atol=1e-12)
try:
    fit_line(np.array([1, 2]), np.array([1]))
except ValueError:
    pass
else:
    raise AssertionError('Mismatched targets must fail')
print('PASS: intercept/slope, prediction, rank deficiency, invalid shape')


## 06 / RANDOMNESS & REPRODUCIBILITY: Use a local random generator without changing global state

Generate reproducible integer samples with a local Generator and explicit interval bounds.

### Hints
- **Create an independent generator:** Avoid np.random.seed, which changes legacy global state.
- **Sample integers in [0,population):** The upper bound is exclusive. Sampling here is with replacement.

### Expected behavior
The same seed/configuration repeats the same sequence in a fixed compatible environment. Results have shape [size] and each value lies in [0,population).


In [ ]:
import numpy as np

def sample_ids(seed: int, population: int, size: int) -> np.ndarray:
    if population <= 0 or size < 0:
        raise ValueError("population must be positive and size nonnegative")
    rng = ...
    return ...


In [ ]:
before = np.random.get_state()
a = sample_ids(23, 100, 12)
b = sample_ids(23, 100, 12)
np.testing.assert_array_equal(a, b)
assert a.shape == (12,) and np.issubdtype(a.dtype, np.integer)
assert ((a >= 0) & (a < 100)).all()
after = np.random.get_state()
assert all(np.array_equal(left, right) for left, right in zip(before, after))
assert sample_ids(23, 100, 0).shape == (0,)
try:
    sample_ids(23, 0, 1)
except ValueError:
    pass
else:
    raise AssertionError('Invalid population must fail')
print('PASS: local reproducibility, bounds, global-state isolation, empty/invalid sizes')


## 07 / ADVANCED TENSOR AXES: Contract features without mixing batch dimensions

Produce [B,T,U] dot products from Q[B,T,D] and K[B,U,D], keeping batch and sequence roles distinct.

### Hints
- **Sum D while preserving B, T, and U:** b appears on both inputs and the output; d is the only contracted axis.

### Expected behavior
Q shaped [2,2,3] and K shaped [2,3,3] produce [2,2,3], with no cross-batch comparisons.


In [ ]:
import numpy as np

def batched_scores(q: np.ndarray, k: np.ndarray) -> np.ndarray:
    if (q.ndim != 3 or k.ndim != 3 or q.shape[0] != k.shape[0]
            or q.shape[2] != k.shape[2]
            or not np.isfinite(q).all() or not np.isfinite(k).all()):
        raise ValueError("Expected finite compatible [B,T,D] and [B,U,D] arrays")
    return ...


In [ ]:
q = np.arange(12., dtype=np.float64).reshape(2, 2, 3)
k = np.arange(18., dtype=np.float64).reshape(2, 3, 3)
result = batched_scores(q, k)
reference = (q[:, :, None, :] * k[:, None, :, :]).sum(axis=-1)
assert result.shape == (2, 2, 3)
np.testing.assert_allclose(result, reference)
changed = k.copy()
changed[1] += 100
np.testing.assert_allclose(batched_scores(q, changed)[0], result[0])
try:
    batched_scores(q, np.ones((3, 3, 3)))
except ValueError:
    pass
else:
    raise AssertionError('Batch mismatch must fail')
print('PASS: axis contraction, independent reference, batch isolation, invalid shapes')


## 08 / PREPROCESSING CONTRACTS: Fit NaN-aware medians without hiding invalid columns

Impute training NaNs using per-column medians while rejecting infinity and all-missing features.

### Hints
- **Fit one NaN-aware training median per feature:** Reduce the sample axis; reject all-missing columns before reduction.
- **Replace only NaNs, broadcasting medians across rows:** Use np.isnan, not x == np.nan. Preserve observed entries and return new storage.

### Expected behavior
For [[1,NaN,4],[3,6,NaN],[NaN,10,8]], medians are [2,8,6] and filled data is [[1,8,4],[3,6,6],[2,10,8]].


In [ ]:
import numpy as np

def fit_impute(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or 0 in x.shape or np.isinf(x).any():
        raise ValueError("Expected nonempty [N,D] data without infinity")
    if np.isnan(x).all(axis=0).any():
        raise ValueError("Cannot fit a median for an all-missing feature")
    medians = ...
    filled = ...
    return filled, medians


In [ ]:
x = np.array([[1., np.nan, 4.], [3., 6., np.nan], [np.nan, 10., 8.]])
original = x.copy()
filled, medians = fit_impute(x)
np.testing.assert_array_equal(medians, [2, 8, 6])
np.testing.assert_array_equal(filled, [[1, 8, 4], [3, 6, 6], [2, 10, 8]])
assert np.array_equal(x, original, equal_nan=True)
assert not np.shares_memory(x, filled)
for invalid in [np.array([[np.nan], [np.nan]]), np.array([[np.inf]])]:
    try:
        fit_impute(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError('Unusable feature values must fail')
print('PASS: medians, broadcasting, unchanged data, independent output, explicit errors')


# Reference solutions

Try the exercises first. Equivalent implementations can be valid even when the browser's limited reference matcher does not accept them.


## Distinguish slices, paired indices, and rectangular selection

```python
import numpy as np

def select_regions(a: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if a.shape != (4, 5):
        raise ValueError("This fixture requires shape (4,5)")
    sliced = a[1:4:2, 1:5:2]
    paired = a[[0, 3], [1, 4]]
    rectangle = a[np.ix_([0, 3], [1, 4])]
    return sliced, paired, rectangle
```

Basic slicing and advanced indexing have different ownership semantics. Coordinate arrays must broadcast; lengths 3 and 2 do not broadcast as a pair. Use np.ix_ when you want the Cartesian product.


## Standardize features with the correct reduction axis

```python
import numpy as np

def fit_standardize(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or 0 in x.shape or not np.isfinite(x).all():
        raise ValueError("Expected a finite nonempty [samples,features] array")
    mean = x.mean(axis=0, keepdims=True)
    scale = x.std(axis=0, keepdims=True)
    scale = np.where(scale == 0, 1.0, scale)
    standardized = (x - mean) / scale
    return standardized, mean, scale
```

Constant-feature handling is an explicit policy, not an accidental divide-by-zero fallback. Fit statistics only on training data and reuse them for validation/test; recomputing on held-out data leaks information.


## Compute stable row probabilities in NumPy

```python
import numpy as np

def row_softmax(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    if logits.ndim < 1 or 0 in logits.shape or not np.isfinite(logits).all():
        raise ValueError("Expected finite, nonempty logits")
    shifted = logits - logits.max(axis=-1, keepdims=True)
    weights = np.exp(shifted)
    return weights / weights.sum(axis=-1, keepdims=True)
```

This exercise requires finite, nonempty inputs and rejects masked infinities. Attention with forbidden keys needs a separate mask/all-masked-row contract. NumPy computes values, not an automatic gradient graph.


## Vectorize pairwise squared distances

```python
import numpy as np

def squared_distances(x: np.ndarray, centers: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    centers = np.asarray(centers, dtype=np.float64)
    if (x.ndim != 2 or centers.ndim != 2 or 0 in x.shape or 0 in centers.shape
            or x.shape[1] != centers.shape[1]
            or not np.isfinite(x).all() or not np.isfinite(centers).all()):
        raise ValueError("Expected finite nonempty [N,D] and [K,D] arrays")
    difference = x[:, None, :] - centers[None, :, :]
    return np.sum(difference ** 2, axis=-1)
```

Vectorization removes Python loops, not memory costs: the difference array has N*K*D elements. Chunk large problems or consider the norm/matmul identity with cancellation safeguards. Float64 conversion avoids integer arithmetic overflow for these fixtures, not arbitrary floating-point overflow.


## Fit a line without explicitly inverting a matrix

```python
import numpy as np

def fit_line(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    if (x.ndim != 1 or y.shape != x.shape or x.size < 2
            or not np.isfinite(x).all() or not np.isfinite(y).all()):
        raise ValueError("Expected matching finite vectors with at least two samples")
    design = np.column_stack([np.ones_like(x), x])
    coefficients = np.linalg.lstsq(design, y, rcond=None)[0]
    return coefficients
```

Explicit inversion of the normal equations is unnecessary and can worsen conditioning. lstsq handles rectangular/rank-deficient designs; a square nonsingular system may instead use solve. Inspect rank, residuals and singular values in a real analysis.


## Use a local random generator without changing global state

```python
import numpy as np

def sample_ids(seed: int, population: int, size: int) -> np.ndarray:
    if population <= 0 or size < 0:
        raise ValueError("population must be positive and size nonnegative")
    rng = np.random.default_rng(seed)
    return rng.integers(0, population, size=size)
```

A seed is part of reproducibility, not a universal cross-version/hardware guarantee. Record generator, versions, call order and data order. Independent generators avoid disturbing randomness elsewhere in a program.


## Contract features without mixing batch dimensions

```python
import numpy as np

def batched_scores(q: np.ndarray, k: np.ndarray) -> np.ndarray:
    if (q.ndim != 3 or k.ndim != 3 or q.shape[0] != k.shape[0]
            or q.shape[2] != k.shape[2]
            or not np.isfinite(q).all() or not np.isfinite(k).all()):
        raise ValueError("Expected finite compatible [B,T,D] and [B,U,D] arrays")
    return np.einsum("btd,bud->btu", q, k)
```

einsum specifies axes explicitly; it is not automatically faster than matmul. Array.T reverses every axis for an n-D array, so use swapaxes on the last two dimensions when that is the intended batched transpose. Numerical accuracy still depends on dtype.


## Fit NaN-aware medians without hiding invalid columns

```python
import numpy as np

def fit_impute(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or 0 in x.shape or np.isinf(x).any():
        raise ValueError("Expected nonempty [N,D] data without infinity")
    if np.isnan(x).all(axis=0).any():
        raise ValueError("Cannot fit a median for an all-missing feature")
    medians = np.nanmedian(x, axis=0)
    filled = np.where(np.isnan(x), medians, x)
    return filled, medians
```

NaN-aware functions do not decide a safe policy for an all-missing feature. This exercise fails explicitly rather than inventing a value. Fit only on training data; reuse the fitted medians for held-out transformations.
